In [ ]:
import os
import random
import torch
import pandas as pd
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling, 
)



from tqdm import tqdm
import re


: 

In [ ]:

random.seed(42)
torch.manual_seed(42)
np.random.seed(42)

print("Loading GSM8K dataset...")
gsm8k = load_dataset("gsm8k", "main")
train_data = gsm8k["train"]
eval_data = gsm8k["test"] 

model_name = "microsoft/phi-3-mini-4k-instruct"
print(f"Loading tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token


In [ ]:

def random_subset_selection(dataset, k):
    indices = random.sample(range(len(dataset)), k)
    return dataset.select(indices)

def preprocess_function(examples):
    inputs = []
    targets = []
    
    for question, answer in zip(examples["question"], examples["answer"]):

        prompt = f"<|user|>\nSolve this math problem step by step:\n{question}\n<|assistant|>\n"
        inputs.append(prompt)
        targets.append(answer)
    
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=512, truncation=True, padding="max_length")
    

    model_inputs["labels"] = labels["input_ids"].copy()
    

    for i in range(len(model_inputs["labels"])):
        model_inputs["labels"][i] = [label if label != tokenizer.pad_token_id else -100 for label in model_inputs["labels"][i]]
    
    return model_inputs


def extract_answer(text):

    numbers = re.findall(r'\d+', text)
    if numbers:
        return numbers[-1]
    return ""

def evaluate_gsm8k(model, tokenizer, eval_dataset, num_samples=100):
   
    if num_samples < len(eval_dataset):
        eval_subset = eval_dataset.select(range(num_samples))
    else:
        eval_subset = eval_dataset
    
    correct = 0
    total = 0
    
    model.eval()
    for example in tqdm(eval_subset, desc="Evaluating"):
        question = example["question"]
        gold_answer = example["answer"]
        
       
        gold_number = extract_answer(gold_answer)
     
        prompt = f"<|user|>\nSolve this math problem step by step:\n{question}\n<|assistant|>\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
   
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                num_beams=1,
            )
        
  
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        generated_text = generated_text[len(prompt):] 
        predicted_number = extract_answer(generated_text)

        if predicted_number == gold_number and gold_number != "":
            correct += 1
        total += 1
    
    accuracy = correct / total if total > 0 else 0
    return accuracy


def finetune_and_evaluate(subset, eval_data, subset_size, results):
    print(f"\n--- Fine-tuning with subset size: {subset_size} ---")
    
    print("Preprocessing data...")
    train_dataset = subset.map(
        preprocess_function, 
        batched=True, 
        remove_columns=subset.column_names,
        desc="Preprocessing training data"
    )
    

    print("Loading model...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,  
        device_map="auto",
        trust_remote_code=True,
    )
    
 
    training_args = TrainingArguments(
        output_dir=f"./phi3_gsm8k_full_{subset_size}",
        per_device_train_batch_size=1, 
        gradient_accumulation_steps=8, 
        learning_rate=5e-6,  
        num_train_epochs=1,  
        logging_steps=10,
        save_strategy="no",  
        fp16=False,  
        bf16=True,
        report_to="none",
        push_to_hub=False,
        optim="adamw_torch",
        weight_decay=0.01,
        max_grad_norm=1.0,
        warmup_ratio=0.03,
    )
    

    data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    return_tensors="pt",
)

    

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    

    print("Starting fine-tuning...")
    trainer.train()
    

    print("Evaluating model...")
    accuracy = evaluate_gsm8k(model, tokenizer, eval_data, num_samples=100)
    
    print(f"Subset size: {subset_size} | Accuracy: {accuracy:.4f}")
    results.append({"subset_size": subset_size, "accuracy": accuracy})
    

    del model
    torch.cuda.empty_cache()
    
    return results


subset_sizes = list(range(900, 2000, 200))
results = []

for size in subset_sizes:
    subset = random_subset_selection(train_data, size)
    results = finetune_and_evaluate(subset, eval_data, size, results)
    
    df = pd.DataFrame(results)
    df.to_csv(f"random_subset_phi3_results_checkpoint_{size}.csv", index=False)


df = pd.DataFrame(results)
df.to_csv("random_subset_phi3_results_full.csv", index=False)
print("\nResults saved to random_subset_phi3_results_full.csv")
print(df)


###  DPP Subset Selection for GSM8K


In [ ]:
import random
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd


gsm8k = load_dataset("gsm8k", "main")
train_data = gsm8k["train"]
eval_data = gsm8k["test"]


def get_embeddings(texts, model_name="sentence-transformers/all-MiniLM-L6-v2", batch_size=32, device='cuda'):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=128).to(device)
            model_output = model(**encoded)

            attention_mask = encoded['attention_mask']
            token_embeddings = model_output.last_hidden_state
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = input_mask_expanded.sum(1)
            batch_embeddings = sum_embeddings / sum_mask
            embeddings.append(batch_embeddings.cpu().numpy())
    return np.vstack(embeddings)

print("Computing embeddings for all training questions...")
question_texts = [q for q in train_data['question']]
embeddings = get_embeddings(question_texts, device='cuda' if torch.cuda.is_available() else 'cpu')


from sklearn.preprocessing import normalize
embeddings = normalize(embeddings, axis=1)
L = np.dot(embeddings, embeddings.T)


def dpp_greedy(L, k):
    """Greedy DPP subset selection."""
    n = L.shape[0]
    selected = []
    cis = np.zeros((k, n))
    di2s = np.copy(np.diag(L))
    for i in range(k):
        if i == 0:
            item = np.argmax(di2s)
            selected.append(item)
            continue
        ci_opt = cis[:i, :]
        di_opt = di2s
        item = np.argmax(di_opt)
        selected.append(item)
        if i == k - 1:
            break
        vi = L[item, :] - np.dot(ci_opt.T, ci_opt[:, item])
        norm = np.sqrt(di2s[item])
        if norm > 1e-10:
            ci = vi / norm
        else:
            ci = vi
        cis[i, :] = ci
        di2s -= ci ** 2
    return selected


subset_sizes = list(range(900, 2000, 200))
results = []

for size in subset_sizes:
    print(f"Selecting DPP subset of size {size}...")
    idxs = dpp_greedy(L, size)
    subset = train_data.select(idxs)

    results.append({'subset_size': size, 'indices': idxs})


df = pd.DataFrame({
    "subset_size": [r['subset_size'] for r in results],
    "indices": [list(map(int, r['indices'])) for r in results]
})
df.to_csv("dpp_subset_indices.csv", index=False)
print("DPP subset indices saved to dpp_subset_indices.csv")


KeyboardInterrupt: 

In [ ]:
import random
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    AutoModel,
)
from tqdm import tqdm
import re
from sklearn.preprocessing import normalize


random.seed(42)
torch.manual_seed(42)
np.random.seed(42)


print("Loading GSM8K dataset...")
gsm8k = load_dataset("gsm8k", "main")
train_data = gsm8k["train"]
eval_data = gsm8k["test"]


model_name = "microsoft/phi-3-mini-4k-instruct"
print(f"Loading tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token


def get_embeddings(texts, model_name="sentence-transformers/all-MiniLM-L6-v2", batch_size=32, device='cuda'):
    emb_tokenizer = AutoTokenizer.from_pretrained(model_name)
    emb_model = AutoModel.from_pretrained(model_name).to(device)
    emb_model.eval()
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            encoded = emb_tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=128).to(device)
            model_output = emb_model(**encoded)

            attention_mask = encoded['attention_mask']
            token_embeddings = model_output.last_hidden_state
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = input_mask_expanded.sum(1)
            batch_embeddings = sum_embeddings / sum_mask
            embeddings.append(batch_embeddings.cpu().numpy())
    return np.vstack(embeddings)


def dpp_greedy(L, k):
    n = L.shape[0]
    selected = []
    cis = np.zeros((k, n))
    di2s = np.copy(np.diag(L))
    for i in range(k):
        if i == 0:
            item = np.argmax(di2s)
            selected.append(item)
            continue
        ci_opt = cis[:i, :]
        di_opt = di2s
        item = np.argmax(di_opt)
        selected.append(item)
        if i == k - 1:
            break
        vi = L[item, :] - np.dot(ci_opt.T, ci_opt[:, item])
        norm = np.sqrt(di2s[item])
        if norm > 1e-10:
            ci = vi / norm
        else:
            ci = vi
        cis[i, :] = ci
        di2s -= ci ** 2
    return selected


def dpp_subset_selection(dataset, k, embeddings=None):
    if embeddings is None:
        texts = [q for q in dataset['question']]
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        embeddings = get_embeddings(texts, device=device)
    embeddings = normalize(embeddings, axis=1)
    L = np.dot(embeddings, embeddings.T)
    idxs = dpp_greedy(L, k)
    return dataset.select(idxs)


def preprocess_function(examples):
    inputs = []
    targets = []
    for question, answer in zip(examples["question"], examples["answer"]):
        prompt = f"<|user|>\nSolve this math problem step by step:\n{question}\n<|assistant|>\n"
        inputs.append(prompt)
        targets.append(answer)
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True, padding="max_length")
    labels = tokenizer(targets, max_length=512, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"].copy()
    for i in range(len(model_inputs["labels"])):
        model_inputs["labels"][i] = [label if label != tokenizer.pad_token_id else -100 for label in model_inputs["labels"][i]]
    return model_inputs

def extract_answer(text):
    numbers = re.findall(r'\d+', text)
    if numbers:
        return numbers[-1]
    return ""

def evaluate_gsm8k(model, tokenizer, eval_dataset, num_samples=100):
    if num_samples < len(eval_dataset):
        eval_subset = eval_dataset.select(range(num_samples))
    else:
        eval_subset = eval_dataset
    correct = 0
    total = 0
    model.eval()
    for example in tqdm(eval_subset, desc="Evaluating"):
        question = example["question"]
        gold_answer = example["answer"]
        gold_number = extract_answer(gold_answer)
        prompt = f"<|user|>\nSolve this math problem step by step:\n{question}\n<|assistant|>\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                num_beams=1,
            )
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        generated_text = generated_text[len(prompt):]
        predicted_number = extract_answer(generated_text)
        if predicted_number == gold_number and gold_number != "":
            correct += 1
        total += 1
    accuracy = correct / total if total > 0 else 0
    return accuracy

def finetune_and_evaluate(subset, eval_data, subset_size, results):
    print(f"\n--- Fine-tuning with DPP subset size: {subset_size} ---")
    print("Preprocessing data...")
    train_dataset = subset.map(
        preprocess_function, 
        batched=True, 
        remove_columns=subset.column_names,
        desc="Preprocessing training data"
    )
    print("Loading model...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    training_args = TrainingArguments(
        output_dir=f"./phi3_gsm8k_dpp_{subset_size}",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=5e-6,
        num_train_epochs=1,
        logging_steps=10,
        save_strategy="no",
        fp16=False,
        bf16=True,
        report_to="none",
        push_to_hub=False,
        optim="adamw_torch",
        weight_decay=0.01,
        max_grad_norm=1.0,
        warmup_ratio=0.03,
    )
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,
        return_tensors="pt",
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    print("Starting fine-tuning...")
    trainer.train()
    print("Evaluating model...")
    accuracy = evaluate_gsm8k(model, tokenizer, eval_data, num_samples=100)
    print(f"Subset size: {subset_size} | Accuracy: {accuracy:.4f}")
    results.append({"subset_size": subset_size, "accuracy": accuracy})
    del model
    torch.cuda.empty_cache()
    return results


subset_sizes = list(range(900, 2000, 200))
results = []


print("Computing embeddings for DPP selection...")
all_embeddings = get_embeddings([q for q in train_data['question']], device='cuda' if torch.cuda.is_available() else 'cpu')

for size in subset_sizes:
    subset = dpp_subset_selection(train_data, size, embeddings=all_embeddings)
    results = finetune_and_evaluate(subset, eval_data, size, results)
    df = pd.DataFrame(results)
    df.to_csv(f"dpp_subset_phi3_results_checkpoint_{size}.csv", index=False)

df = pd.DataFrame(results)
df.to_csv("dpp_subset_phi3_results_full.csv", index=False)
print("\nResults saved to dpp_subset_phi3_results_full.csv")
print(df)



2025-05-02 02:57:24.236216: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-02 02:57:24.247571: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746134844.260749 1491810 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746134844.264514 1491810 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-02 02:57:24.278222: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

Loading GSM8K dataset...
Loading tokenizer for microsoft/phi-3-mini-4k-instruct...
Computing embeddings for DPP selection...

--- Fine-tuning with DPP subset size: 900 ---
Preprocessing data...


Preprocessing training data:   0%|          | 0/900 [00:00<?, ? examples/s]

Loading model...


`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


...

Starting fine-tuning...


Step,Training Loss
10,8.457400
20,3.943500
30,1.080900
40,0.463900
50,0.462700
60,0.465500
70,0.525600
80,0.475200
90,0.525000
100,0.325800


Evaluating model...


Evaluating: 100%|██████████| 100/100 [09:05<00:00,  5.46s/it]

Subset size: 1900 | Accuracy: 0.4500

Results saved to dpp_subset_phi3_results_full.csv
   subset_size  accuracy
0          900      0.49
1         1100      0.44
2         1300      0.45
3         1500      0.47
4         1700      0.43
5         1900      0.45
